# 04 · Evaluation, and what the numbers actually mean

Three layers: automatic question metrics, backend reliability, and a
37-participant user study.

The study is where the project's assumptions got tested against real people, and
it is where the most uncomfortable finding lives.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)

SAMPLE = REPO_ROOT / "data" / "sample"
RESULTS = REPO_ROOT / "results"
print("Repository root:", REPO_ROOT.name)


Repository root: portfolio-clean


## Choosing the vision model

Notebook 03 established that conditioning on the image beats conditioning on a
text description of it. That leaves the question of *which* multimodal model.

Three Gemini variants were compared on identical crops and identical prompts,
scored by BERTScore against 25 hand-written reference questions (5 per amenity).

In [2]:
bert = pd.read_csv(
    RESULTS / "analysis_summary" / "model_comparison" / "bertscore_by_amenity.csv"
)
fallback_models = pd.read_csv(
    RESULTS / "analysis_summary" / "model_comparison" / "model_fallback_rates_summary.csv",
    encoding="utf-8-sig",
)

comparison = bert.pivot(index="model", columns="amenity", values="bertscore")
comparison["overall"] = comparison.mean(axis=1).round(3)
comparison.sort_values("overall", ascending=False)

amenity,bathtub,hairdryer,kettle,mirror,tv,overall
model,,,,,,
Gemini 3.0 Pro Preview,0.719,0.694,0.728,0.698,0.697,0.707
Gemini 2.5 Flash,0.705,0.717,0.704,0.686,0.686,0.700
Gemini 2.5 Pro,0.708,0.684,0.707,0.696,0.678,0.695


In [3]:
fallback_models

,Model,Total Questions,Fallbacks,Fallback Rate (%)
0,Gemini 2.5 Flash,7770,733,9.43
1,Gemini 2.5 Pro,7770,780,10.04
2,Gemini 3.0 Pro,7770,610,7.85


![Model selection](../results/figures/generation/model_selection.png)

Gemini 3.0 Pro Preview wins on both measures, so it was used for everything
downstream. But the honest reading is that **the spread is under 2% on
BERTScore**, so model choice is nearly irrelevant at this scale, and the fallback
rate is the more useful discriminator. Reporting only the winner would have
implied a decisiveness the data does not support.

## Question quality

Two measures, both computed over the full Gemini 3.0 Pro Preview output.

**Semantic diversity** is measured *within* each image's question set: one minus
the mean pairwise cosine similarity between the questions generated for a single
detection. It asks whether the model varies its questions about one photo, a
different and more demanding question than whether it varies across photos.

In [4]:
diversity = pd.read_csv(RESULTS / "analysis_summary" / "semantic_diversity_by_amenity.csv")
fallback_amenity = pd.read_csv(
    RESULTS / "analysis_summary" / "model_comparison" / "gemini30_fallback_rates.csv",
    encoding="utf-8-sig",
)

display(diversity)
display(fallback_amenity)

,amenity,mean_similarity,mean_semantic_diversity
0,bathtub,0.285,0.715
1,tv,0.292,0.708
2,kettle,0.316,0.684
3,hairdryer,0.191,0.809
4,mirror,0.372,0.628
5,overall,0.291,0.709


,Amenity,Total Questions,Fallback Questions,Fallback Rate (%)
0,Bathtub,1380,156,11.30
1,Tv,1335,9,0.67
2,Hairdryer,1005,26,2.59
3,Mirror,3135,341,10.88
4,Kettle,915,78,8.52
5,TOTAL,7770,610,7.85


![Question quality by amenity](../results/figures/generation/generation_quality.png)

Mean semantic diversity is **0.709**, ranging from 0.628 for mirror to 0.809 for
hairdryer. The model is not asking the same thing five times about one
photograph.

The fallback pattern is the more interesting half. TV fails on **0.67%** of
questions; bathtub on **11.30%**, mirror on **10.88%**. Televisions are
standardised dark rectangles. Bathtubs vary in shape and framing; mirrors reflect
the rest of the room and are frequently ambiguous about where the object even
ends.

**Generation robustness tracks visual regularity.** That points at per-amenity
prompt work rather than a global model swap. The failure is concentrated, not
diffuse.

## Does traveller-profile conditioning actually work?

The pipeline conditions generation on who is travelling. Worth checking whether
the model attends to that or quietly ignores it, which is the usual outcome when
a conditioning signal is weak.

In [5]:
distinctiveness = pd.read_csv(
    RESULTS / "analysis_summary" / "profile_distinctiveness.csv",
    encoding="utf-8-sig",
)
distinctiveness.pivot(index="amenity", columns="pair", values="distinctiveness").round(3)

pair,couple_vs_group,single_vs_couple,single_vs_group
amenity,,,
bathtub,0.613,0.496,0.612
hairdryer,0.725,0.679,0.710
kettle,0.652,0.616,0.699
mirror,0.569,0.614,0.645
tv,0.604,0.565,0.655


![Profile conditioning](../results/figures/generation/profile_conditioning.png)

Every profile pair separates by **0.50 to 0.73** cosine distance. The
conditioning lands. Single-vs-group separates most on average (0.664), which is
the pair you would expect to diverge: a solo traveller and a family group want
genuinely different things from the same bathtub. It is worth noting the
exception, though. For bathtub and hairdryer, couple-vs-group separates more.

## The user study

37 participants rated 75 generated questions 1–5 on how helpful each would be
when choosing a hotel. Full protocol, demographics, and limitations:
[`docs/user_study_summary.md`](../docs/user_study_summary.md).

**Overall mean: 3.36 / 5.** That is a moderate result and is reported as one.
87% of questions cleared "moderately helpful"; only 37% reached 3.5+.

In [6]:
study = pd.DataFrame({
    "amenity": ["Bathtub", "Kettle", "Hairdryer", "TV", "Mirror"],
    "mean": [3.45, 3.45, 3.41, 3.28, 3.22],
    "worst question": [2.76, 3.05, 2.84, 2.19, 2.78],
    "best question": [3.86, 4.08, 3.73, 4.03, 3.62],
})
study["spread"] = (study["best question"] - study["worst question"]).round(2)
study

,amenity,mean,worst question,best question,spread
0,Bathtub,3.45,2.76,3.86,1.10
1,Kettle,3.45,3.05,4.08,1.03
2,Hairdryer,3.41,2.84,3.73,0.89
3,TV,3.28,2.19,4.03,1.84
4,Mirror,3.22,2.78,3.62,0.84


## The finding that matters

Look at the `spread` column, not the `mean` column.

The gap *between* amenities is small, 3.22 to 3.45, which is noise at this
sample size. The gap *within* each amenity is large. TV ranges from 2.19 to 4.03
on the same amenity, from the same pipeline.

So the interesting question is not "which amenity works best" but "what separates
a good question from a bad one".

In [7]:
best_worst = pd.DataFrame({
    "rating": [4.08, 4.03, 3.92, 2.78, 2.22, 2.19],
    "question is about": [
        "whether the appliance looked modern and clean",
        "whether the TV supported streaming apps",
        "whether guests could use their own streaming accounts",
        "whether a mirror was framed or frameless",
        "whether a white border was a display effect or a bezel",
        "whether a TV's screen edge was physical or rendered",
    ],
    "kind": [
        "booking decision", "booking decision", "booking decision",
        "visual detail", "visual ambiguity", "visual ambiguity",
    ],
})
best_worst

,rating,question is about,kind
0,4.08,whether the appliance looked modern and clean,booking decision
1,4.03,whether the TV supported streaming apps,booking decision
2,3.92,whether guests could use their own streaming accounts,booking decision
3,2.78,whether a mirror was framed or frameless,visual detail
4,2.22,whether a white border was a display effect or a bezel,visual ambiguity
5,2.19,whether a TV's screen edge was physical or rendered,visual ambiguity


**Participants rated questions about booking decisions highly, and questions
about visual ambiguity poorly.**

The low-scoring questions are exactly the ones a detector-driven system produces
naturally. When a model is uncertain whether that white border is a bezel or a
display artefact, the obvious move is to ask about it. Every participant rating
says: nobody cares.

**Detection uncertainty and traveller relevance are different quantities.** A
system that generates questions about whatever the model found ambiguous is
optimising the wrong objective, and it will feel broken to users while its
metrics look fine.

That reframes the whole pipeline. The useful signal is not "what is the detector
unsure about" but "what would change someone's booking decision", and those are
close to uncorrelated.

## Where the evidence is weak

Stated plainly, because it bounds everything above.

- **No control condition.** Participants never rated human-written or randomly
  selected questions. 3.36/5 has no reference point; it could be excellent or
  mediocre and this study cannot distinguish them. This is the single biggest
  weakness of the evaluation design.
- **Small, skewed sample.** 37 people, nearly 60% aged 18–25.
- **Stated preference, not behaviour.** Nobody booked anything. Rating a question
  as helpful is not evidence it helps.
- **Prompt and threshold were selected on the test split**, so the reported
  figures are best-of-sweep and optimistic.
- **A small part of the TV set is stock photography** rather than accommodation
  listings, so TV is slightly less domain-consistent than the other four.
- **The vision-conditioned questions were never rated at all.** The study covered
  text-only output, so the diversity gain in notebook 03 has no helpfulness
  evidence behind it.

## What I would do next

The ablation this project stops short of: the **same model, with and without the
cropped region**, everything else fixed. That separates conditioning from model
capability and would turn Finding 3 from suggestive into conclusive.

Then re-run the study with two additions: a human-written control condition, and
the vision-generated questions included. That gives the helpfulness scores a
reference point and tests whether diversity actually buys relevance.

And on the detection side: per-amenity thresholds calibrated against the cost of
a wrong question, rather than a single F1-maximising number. The user study
suggests a false positive is expensive in a way F1 does not capture.